# 06 – Benign Control Dataset

This notebook builds the benign control dataset used to measure false positives in the later defense experiments.

Primary source:

```text
axiotic/ogma-prompt-injection-benign
```

The goal is not to use every benign example. Instead, we construct a compact and reproducible control set with two groups:

- `general` – ordinary benign requests
- `instruction_heavy` – benign prompts that contain code, commands, long context, or strongly instruction-like content

The second group is especially important because an overly aggressive prompt-injection defense may incorrectly block legitimate but instruction-heavy inputs.

The initial target size is:

```text
100 general benign prompts
100 instruction-heavy benign prompts
```

The notebook keeps the sampling procedure reproducible and saves the selected controls as CSV.


In [1]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

# Run once if needed:
# %pip install datasets pandas

from pathlib import Path
import hashlib
import re

import pandas as pd
from datasets import load_dataset


c:\Users\zalan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
#  2. BLOCK – PATHS AND CONFIGURATION
# =========================

PROJECT_ROOT = Path.cwd().resolve()

DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

RAW_DATA_DIR = (
    DATA_DIR
    / "raw"
)

FINAL_DATA_DIR = (
    DATA_DIR
    / "final"
)

RAW_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DATASET_NAME = (
    "axiotic/ogma-prompt-injection-benign"
)

RAW_CSV_PATH = (
    RAW_DATA_DIR
    / "ogma_benign_raw.csv"
)

FINAL_BENIGN_PATH = (
    FINAL_DATA_DIR
    / "benign_control_dataset.csv"
)

SUMMARY_PATH = (
    FINAL_DATA_DIR
    / "benign_control_summary.csv"
)

GENERAL_SAMPLE_SIZE = 100
INSTRUCTION_HEAVY_SAMPLE_SIZE = 100
RANDOM_STATE = 42

FORCE_DOWNLOAD = False

print("Dataset:", DATASET_NAME)
print("Raw cache:", RAW_CSV_PATH)
print("Final dataset:", FINAL_BENIGN_PATH)


Dataset: axiotic/ogma-prompt-injection-benign
Raw cache: C:\work\LLM filtering\TDK\code\data\raw\ogma_benign_raw.csv
Final dataset: C:\work\LLM filtering\TDK\code\data\final\benign_control_dataset.csv


In [3]:
# =========================
#  3. BLOCK – DOWNLOAD DATASET OR LOAD CACHE
# =========================

if (
    RAW_CSV_PATH.exists()
    and not FORCE_DOWNLOAD
):
    print(
        "Loading cached raw dataset:"
    )
    print(
        RAW_CSV_PATH
    )

    df_raw = pd.read_csv(
        RAW_CSV_PATH
    )

else:
    print(
        "Downloading dataset from Hugging Face:"
    )
    print(
        DATASET_NAME
    )

    dataset = load_dataset(
        DATASET_NAME
    )

    print(
        "\nAvailable splits:"
    )

    print(
        dataset
    )

    frames = []

    for split_name, split in dataset.items():
        df_split = (
            split
            .to_pandas()
            .copy()
        )

        df_split[
            "dataset_split"
        ] = split_name

        frames.append(
            df_split
        )

    df_raw = pd.concat(
        frames,
        ignore_index=True,
    )

    df_raw.to_csv(
        RAW_CSV_PATH,
        index=False,
    )

    print(
        "\nRaw dataset saved:"
    )

    print(
        "-",
        RAW_CSV_PATH
    )

print(
    "\nRaw dataset shape:",
    df_raw.shape
)


axiotic/ogma-prompt-injection-benign


Generating train split: 100%|██████████| 11368/11368 [00:00<00:00, 137935.08 examples/s]


Available splits:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'form', 'length', 'task_type', 'domain', 'subtype'],
        num_rows: 11368
    })
})

Raw dataset saved:
- C:\work\LLM filtering\TDK\code\data\raw\ogma_benign_raw.csv

Raw dataset shape: (11368, 8)


In [4]:
# =========================
#  4. BLOCK – INSPECT DATASET SCHEMA
# =========================

print(
    "Columns:"
)

for column in df_raw.columns:
    print(
        "-",
        column
    )

print(
    "\nSample rows:"
)

display(
    df_raw.head()
)


Columns:
- text
- label
- form
- length
- task_type
- domain
- subtype
- dataset_split

Sample rows:


,text,label,form,length,task_type,domain,subtype,dataset_split
0,Summarize the main points from that parenting ...,0,statement,short,summarise a piece of text,parenting,natural,train
1,Please summarize best practices for helping ch...,0,statement,short,summarise a piece of text,parenting,natural,train
2,Condense the key takeaways from that research ...,0,statement,short,summarise a piece of text,parenting,natural,train
3,Summarize the guidance document regarding scre...,0,statement,short,summarise a piece of text,parenting,natural,train
4,Please provide a summary of the article about ...,0,statement,short,summarise a piece of text,parenting,natural,train


In [5]:
# =========================
#  5. BLOCK – DETECT TEXT COLUMN
# =========================

candidate_text_columns = [
    "prompt",
    "text",
    "instruction",
    "input",
    "content",
    "query",
]

TEXT_COLUMN = None

for candidate in candidate_text_columns:
    if candidate in df_raw.columns:
        TEXT_COLUMN = candidate
        break

if TEXT_COLUMN is None:
    object_columns = [
        column
        for column in df_raw.columns
        if df_raw[
            column
        ].dtype == "object"
    ]

    if not object_columns:
        raise ValueError(
            "No suitable text column was found."
        )

    average_lengths = {}

    for column in object_columns:
        average_lengths[
            column
        ] = (
            df_raw[
                column
            ]
            .fillna("")
            .astype(str)
            .str.len()
            .mean()
        )

    TEXT_COLUMN = max(
        average_lengths,
        key=average_lengths.get,
    )

print(
    "Selected text column:",
    TEXT_COLUMN
)


Selected text column: text


In [6]:
# =========================
#  6. BLOCK – BASIC CLEANING
# =========================

df_clean = (
    df_raw
    .copy()
)

df_clean[
    "benign_text"
] = (
    df_clean[
        TEXT_COLUMN
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_clean = (
    df_clean[
        df_clean[
            "benign_text"
        ].str.len() > 0
    ]
    .copy()
)

before_duplicates = len(
    df_clean
)

df_clean = (
    df_clean
    .drop_duplicates(
        subset=[
            "benign_text"
        ]
    )
    .reset_index(
        drop=True
    )
)

removed_duplicates = (
    before_duplicates
    - len(df_clean)
)

print(
    "Rows after cleaning:",
    len(df_clean)
)

print(
    "Exact duplicate texts removed:",
    removed_duplicates
)


Rows after cleaning: 11368
Exact duplicate texts removed: 0


In [7]:
# =========================
#  7. BLOCK – ADD BASIC TEXT FEATURES
# =========================

df_clean[
    "char_length"
] = (
    df_clean[
        "benign_text"
    ]
    .str.len()
)

df_clean[
    "word_count"
] = (
    df_clean[
        "benign_text"
    ]
    .str.split()
    .str.len()
)

df_clean[
    "line_count"
] = (
    df_clean[
        "benign_text"
    ]
    .str.count("\n")
    + 1
)

display(
    df_clean[
        [
            "benign_text",
            "char_length",
            "word_count",
            "line_count",
        ]
    ]
    .head()
)


,benign_text,char_length,word_count,line_count
0,Summarize the main points from that parenting ...,83,11,1
1,Please summarize best practices for helping ch...,83,10,1
2,Condense the key takeaways from that research ...,83,12,1
3,Summarize the guidance document regarding scre...,84,10,1
4,Please provide a summary of the article about ...,81,12,1


## Instruction-heavy heuristic

If the source dataset already contains useful subtype metadata, it is used below.

Otherwise, the notebook falls back to a transparent heuristic based on text characteristics.

A prompt is considered potentially `instruction_heavy` when it contains one or more of the following:

- code blocks
- shell or command-line markers
- strong imperative wording
- multiple lines of structured instructions
- unusually long context

This label is only used for benign sampling. It is **not** a security label and does not imply maliciousness.


In [8]:
# =========================
#  8. BLOCK – DETECT AVAILABLE METADATA
# =========================

metadata_columns = [
    column
    for column in [
        "task_type",
        "subtype",
        "category",
        "type",
        "source",
    ]
    if column in df_clean.columns
]

print(
    "Available metadata columns:"
)

for column in metadata_columns:
    print(
        "-",
        column
    )

for column in metadata_columns:
    print(
        f"\nTop values for {column}:"
    )

    display(
        df_clean[
            column
        ]
        .value_counts(
            dropna=False
        )
        .head(20)
    )


Available metadata columns:
- task_type
- subtype

Top values for task_type:


task_type
bare utility command                                                                    496
rewrite or rephrase a sentence                                                          442
convert units or formats                                                                400
create a quiz or practice questions                                                     344
draft an email or letter                                                                338
define a term                                                                           335
compare two things                                                                      317
write a short creative piece                                                            315
outline a plan or document                                                              312
translate a phrase or passage                                                           312
fix grammar or spelling                                               


Top values for subtype:


subtype
natural     7092
abstract    1911
command     1847
code         518
Name: count, dtype: int64

In [9]:
# =========================
#  9. BLOCK – CLASSIFY BENIGN DIFFICULTY GROUP
# =========================

instruction_metadata_keywords = {
    "command",
    "code",
    "shell",
    "terminal",
    "long_with_context",
    "long context",
    "programming",
    "script",
}

def metadata_indicates_instruction_heavy(
    row: pd.Series,
) -> bool:
    for column in metadata_columns:
        value = str(
            row.get(
                column,
                "",
            )
        ).strip().lower()

        if any(
            keyword in value
            for keyword in instruction_metadata_keywords
        ):
            return True

    return False


def text_indicates_instruction_heavy(
    text: str,
) -> bool:
    lowered = text.lower()

    code_or_command_patterns = [
        r"```",
        r"\b(?:bash|powershell|cmd|terminal|shell)\b",
        r"\b(?:python|javascript|java|c\+\+|sql)\b",
        r"\b(?:run|execute|install|compile|write|create|delete|open|save)\b",
        r"^\s*[-*]\s+",
        r"^\s*\d+[.)]\s+",
    ]

    pattern_hits = sum(
        bool(
            re.search(
                pattern,
                lowered,
                flags=re.MULTILINE,
            )
        )
        for pattern in code_or_command_patterns
    )

    long_context = (
        len(text) >= 1200
    )

    many_lines = (
        text.count("\n") >= 5
    )

    return (
        pattern_hits >= 2
        or long_context
        or many_lines
    )


df_clean[
    "metadata_instruction_heavy"
] = (
    df_clean.apply(
        metadata_indicates_instruction_heavy,
        axis=1,
    )
)

df_clean[
    "text_instruction_heavy"
] = (
    df_clean[
        "benign_text"
    ]
    .apply(
        text_indicates_instruction_heavy
    )
)

df_clean[
    "benign_group"
] = "general"

df_clean.loc[
    (
        df_clean[
            "metadata_instruction_heavy"
        ]
        | df_clean[
            "text_instruction_heavy"
        ]
    ),
    "benign_group",
] = "instruction_heavy"

display(
    df_clean[
        "benign_group"
    ]
    .value_counts()
)


benign_group
general              8891
instruction_heavy    2477
Name: count, dtype: int64

In [10]:
# =========================
#  10. BLOCK – SAMPLE GENERAL BENIGN PROMPTS
# =========================

df_general_pool = (
    df_clean[
        df_clean[
            "benign_group"
        ]
        == "general"
    ]
    .copy()
)

if len(
    df_general_pool
) < GENERAL_SAMPLE_SIZE:
    raise ValueError(
        "Not enough general benign prompts "
        "for the requested sample size."
    )

df_general = (
    df_general_pool
    .sample(
        n=GENERAL_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )
    .copy()
)

print(
    "General benign sample:",
    len(df_general)
)


General benign sample: 100


In [11]:
# =========================
#  11. BLOCK – SAMPLE INSTRUCTION-HEAVY BENIGN PROMPTS
# =========================

df_heavy_pool = (
    df_clean[
        df_clean[
            "benign_group"
        ]
        == "instruction_heavy"
    ]
    .copy()
)

if len(
    df_heavy_pool
) < INSTRUCTION_HEAVY_SAMPLE_SIZE:
    raise ValueError(
        "Not enough instruction-heavy benign prompts "
        "for the requested sample size."
    )

df_instruction_heavy = (
    df_heavy_pool
    .sample(
        n=INSTRUCTION_HEAVY_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )
    .copy()
)

print(
    "Instruction-heavy benign sample:",
    len(df_instruction_heavy)
)


Instruction-heavy benign sample: 100


In [12]:
# =========================
#  12. BLOCK – BUILD FINAL BENIGN CONTROL SET
# =========================

df_benign = pd.concat(
    [
        df_general,
        df_instruction_heavy,
    ],
    ignore_index=True,
)

df_benign = (
    df_benign
    .sample(
        frac=1.0,
        random_state=RANDOM_STATE,
    )
    .reset_index(
        drop=True
    )
)

print(
    "Final benign dataset size:",
    len(df_benign)
)

display(
    df_benign[
        "benign_group"
    ]
    .value_counts()
)


Final benign dataset size: 200


benign_group
general              100
instruction_heavy    100
Name: count, dtype: int64

In [13]:
# =========================
#  13. BLOCK – CREATE STABLE BENIGN IDS
# =========================

def create_benign_id(
    text: str,
) -> str:
    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()

    return (
        "B_"
        + digest[:16]
    )


df_benign[
    "benign_id"
] = (
    df_benign[
        "benign_text"
    ]
    .apply(
        create_benign_id
    )
)

if df_benign[
    "benign_id"
].duplicated().any():
    raise ValueError(
        "Duplicate benign IDs detected."
    )

display(
    df_benign[
        [
            "benign_id",
            "benign_group",
            "benign_text",
        ]
    ]
    .head()
)


,benign_id,benign_group,benign_text
0,B_7e8dd09b76117c1a,general,Can you help me change this temperature from F...
1,B_70a51a7a43435db9,general,I'm wondering which would be better for my bat...
2,B_1a28a81827727b63,general,Could you pull out the main points from this e...
3,B_7b38d7b9da0a8bbc,instruction_heavy,go get -u github.com/gorilla/mux@latest
4,B_cc0c29e5e29af299,instruction_heavy,touch ~/projects/pihole_dashboard/init_complet...


In [14]:
# =========================
#  14. BLOCK – FINAL VALIDATION
# =========================

expected_size = (
    GENERAL_SAMPLE_SIZE
    + INSTRUCTION_HEAVY_SAMPLE_SIZE
)

if len(
    df_benign
) != expected_size:
    raise ValueError(
        "Unexpected final benign dataset size."
    )

if df_benign[
    "benign_text"
].isna().any():
    raise ValueError(
        "Missing benign text detected."
    )

if df_benign[
    "benign_text"
].duplicated().any():
    raise ValueError(
        "Duplicate benign texts detected."
    )

group_counts = (
    df_benign[
        "benign_group"
    ]
    .value_counts()
)

print(
    "Validation successful."
)

print(
    "\nGroup distribution:"
)

display(
    group_counts
)


Validation successful.

Group distribution:


benign_group
general              100
instruction_heavy    100
Name: count, dtype: int64

In [15]:
# =========================
#  15. BLOCK – BUILD SUMMARY TABLE
# =========================

summary_rows = []

for group_name, group_df in df_benign.groupby(
    "benign_group"
):
    summary_rows.append(
        {
            "benign_group": group_name,
            "sample_count": len(
                group_df
            ),
            "mean_char_length": (
                group_df[
                    "char_length"
                ].mean()
            ),
            "median_char_length": (
                group_df[
                    "char_length"
                ].median()
            ),
            "mean_word_count": (
                group_df[
                    "word_count"
                ].mean()
            ),
            "median_word_count": (
                group_df[
                    "word_count"
                ].median()
            ),
        }
    )

df_summary = pd.DataFrame(
    summary_rows
)

display(
    df_summary
)


,benign_group,sample_count,mean_char_length,median_char_length,mean_word_count,median_word_count
0,general,100,201.50,92.5,31.34,15.0
1,instruction_heavy,100,149.78,112.0,22.36,18.5


In [16]:
# =========================
#  16. BLOCK – SELECT FINAL OUTPUT COLUMNS
# =========================

preferred_columns = [
    "benign_id",
    "benign_group",
    "benign_text",
    "char_length",
    "word_count",
    "line_count",
]

metadata_output_columns = [
    column
    for column in metadata_columns
    if column not in preferred_columns
]

df_benign_final = (
    df_benign[
        preferred_columns
        + metadata_output_columns
    ]
    .copy()
)

display(
    df_benign_final.head()
)


,benign_id,benign_group,benign_text,char_length,word_count,line_count,task_type,subtype
0,B_7e8dd09b76117c1a,general,Can you help me change this temperature from F...,79,14,1,convert units or formats,natural
1,B_70a51a7a43435db9,general,I'm wondering which would be better for my bat...,115,18,1,compare two things,natural
2,B_1a28a81827727b63,general,Could you pull out the main points from this e...,728,99,1,summarise a piece of text,natural
3,B_7b38d7b9da0a8bbc,instruction_heavy,go get -u github.com/gorilla/mux@latest,39,4,1,"installing or managing packages (npm, pip, apt...",command
4,B_cc0c29e5e29af299,instruction_heavy,touch ~/projects/pihole_dashboard/init_complet...,52,2,1,a scoped file operation on named project files...,command


In [17]:
# =========================
#  17. BLOCK – SAVE BENIGN CONTROL DATASET
# =========================

df_benign_final.to_csv(
    FINAL_BENIGN_PATH,
    index=False,
)

df_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print(
    "Benign control dataset saved:"
)

print(
    "-",
    FINAL_BENIGN_PATH
)

print(
    "Summary saved:"
)

print(
    "-",
    SUMMARY_PATH
)


Benign control dataset saved:
- C:\work\LLM filtering\TDK\code\data\final\benign_control_dataset.csv
Summary saved:
- C:\work\LLM filtering\TDK\code\data\final\benign_control_summary.csv


## Output files

This notebook creates:

```text
data/final/benign_control_dataset.csv
data/final/benign_control_summary.csv
```

The benign control dataset contains two equally sized groups:

```text
general
instruction_heavy
```

These controls will later be used to measure:

- overall false-positive rate
- false-positive rate on ordinary benign prompts
- false-positive rate on instruction-heavy benign prompts

The next notebook will define the defense strategy set used against both malicious and benign inputs.
